In [ ]:
!pip install -q tokenizers transformers tensorflow matplotlib seaborn koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 1-3. Positional Encoding 시각화
# 목표: sin/cos 패턴으로 위치 정보를 벡터에 추가하는 원리를 이해한다
# 사전 설치: pip install matplotlib seaborn koreanize-matplotlib
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 최대 시퀀스 길이: 몇 개의 위치까지 PE를 계산할지
# GPT-2는 1024, GPT-4는 128000 지원
MAX_LEN = 50

# 임베딩 차원: BPE 실습의 D_MODEL과 맞추면 연결성 확인 가능
# Self-Attention 실습의 D_MODEL과 동일하게 설정하는 것을 권장
D_MODEL = 64

# sin/cos 파형을 그릴 차원 인덱스 (0 ~ D_MODEL-1 범위)
# 숫자가 작을수록 빠른 주파수(짧은 파형), 클수록 느린 주파수(완만한 파형)
DIMS_TO_PLOT = [0, 4, 16, 32]

# 코사인 유사도를 비교할 위치 쌍 (position 0 기준)
# (0, 1): 바로 옆 토큰, (0, 25): 25칸 떨어진 토큰
POSITION_PAIRS = [(0, 1), (0, 5), (0, 25), (0, 49)]

# 히트맵 색상 테마
# 'RdBu': 빨강-흰색-파랑, 'viridis', 'coolwarm' 도 가능
HEATMAP_CMAP = "RdBu"

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── Positional Encoding 계산 ─────────────────────────────────
# 공식:
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#
# pos : 토큰의 위치 (0, 1, 2, ...)
# i   : 차원 인덱스 (0, 1, 2, ... d_model/2)
#
# 핵심 아이디어:
#   차원마다 다른 주파수의 sin/cos를 사용
#   -> 각 위치가 고유한 패턴의 벡터를 가짐
#   -> 모델이 상대적 위치 관계를 학습할 수 있음

def positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]   # (max_len, 1)
    dims      = np.arange(d_model)[np.newaxis, :]   # (1, d_model)

    # 분모: 10000^(2i/d_model)
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)

    # 짝수 차원 -> sin, 홀수 차원 -> cos
    pe = np.where(dims % 2 == 0, np.sin(angles), np.cos(angles))
    return pe   # (max_len, d_model)

pe = positional_encoding(MAX_LEN, D_MODEL)
print(f"PE 행렬 shape: {pe.shape}  -> (max_len={MAX_LEN}, d_model={D_MODEL})")
print(f"값 범위: {pe.min():.3f} ~ {pe.max():.3f}  (항상 -1 ~ +1 사이)")

# ── 시각화 1: PE 패턴 히트맵 ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(pe, aspect='auto', cmap=HEATMAP_CMAP, vmin=-1, vmax=1)
axes[0].set_xlabel("임베딩 차원 (d_model)")
axes[0].set_ylabel("토큰 위치 (position)")
axes[0].set_title("Positional Encoding 패턴\n(빨강=+1, 파랑=-1)", fontsize=12)
plt.colorbar(im, ax=axes[0])

# 시각화 2: 특정 차원의 sin/cos 파형
colors = ['#E24B4A', '#1D9E75', '#378ADD', '#EF9F27']
for d, c in zip(DIMS_TO_PLOT, colors):
    axes[1].plot(pe[:, d], label=f"dim {d}", color=c, linewidth=1.5)

axes[1].set_xlabel("토큰 위치 (position)")
axes[1].set_ylabel("PE 값")
axes[1].set_title("차원별 sin/cos 파형\n(차원마다 다른 주파수)", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')

plt.suptitle("Positional Encoding - 위치 정보를 벡터에 추가하는 방법", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("positional_encoding.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 위치 간 코사인 유사도 ─────────────────────────────────────
# 가까운 위치일수록 PE 벡터가 유사해야 함
# -> 모델이 상대적 거리를 학습할 수 있는 근거

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

print("\n위치 간 PE 코사인 유사도:")
print("(가까운 위치일수록 유사한 벡터 -> 모델이 상대적 위치 학습 가능)")
print("-" * 45)
for p1, p2 in POSITION_PAIRS:
    sim = cosine_sim(pe[p1], pe[p2])
    bar = "█" * int(sim * 20)
    print(f"  위치 {p1:2d} <-> 위치 {p2:2d}: {sim:.3f}  {bar}")

# ── 임베딩 + PE 결합 개념 시각화 ─────────────────────────────
print("\n핵심 요약:")
print("  Self-Attention 자체는 순서 정보가 없음 (집합 연산)")
print("  -> PE를 임베딩에 더해 위치 정보 주입")
print("  -> 토큰 벡터 = 의미(임베딩) + 위치(PE)")
print("  -> 오후 Decoder: 이 벡터로 다음 위치의 토큰을 예측")

# ── 추가 실습: D_MODEL 변경 실험 ─────────────────────────────
print("\n" + "=" * 50)
print("추가 확인: 첫 번째 토큰 위치 벡터의 일부")
print("=" * 50)
print(f"  위치 0의 PE 벡터 (처음 8차원): {pe[0, :8].round(3)}")
print(f"  위치 1의 PE 벡터 (처음 8차원): {pe[1, :8].round(3)}")
print(f"  위치 2의 PE 벡터 (처음 8차원): {pe[2, :8].round(3)}")
print("  -> 위치마다 다른 패턴의 벡터를 가짐")